# 🫀 실험 15c — **공유가 이득인가 손해인가**: 다중라벨 헤드 vs 부위별 개별 분류기

**MedKOS / `notebooks/exp15c_shared_vs_percohort.ipynb`** · 퀘스트 `ailab-2026-0015`
**다운로드 없음** — 실험15가 만든 전량 캐시 `ptbxl_12lead_all.npz` 를 그대로 쓴다.

---

## 실험15의 P-1은 **이기게 되어 있는 시합**이었다

실험15에서 부위 7개 전부가 기준선을 이겼다(7/7). 그런데 그 기준선(**A**)은
5-superclass 모델의 **MI 확률**이었다 — **하벽과 전벽을 구별하라고 배운 적이 없는 모델**이다.
부위 전용 검출기가 범용 검출기를 이기는 건 거의 설계상 보장된다.

**진짜 물어야 할 것은 이거다:**

> 부위를 배우게 하되, **여러 부위를 한 헤드로 같이 배우는 것**과
> **부위마다 따로 배우는 것** 중 어느 쪽이 나은가?

이게 **기준선 B**이고, 2단계(라벨 트리 전체)의 설계를 가른다.
소견이 수십 개로 늘 때 **헤드 하나에 다 넣을지, 나눌지**가 여기서 정해진다.

## 이론적으로 양쪽 다 말이 된다

| | 공유(다중라벨 헤드 하나)가 유리한 이유 | 개별(부위별 이진)이 유리한 이유 |
|---|---|---|
| **희소 부위** | 흔한 부위에서 배운 표현을 **전이**받는다 | — |
| **흔한 부위** | — | 용량을 독점한다. 다른 부위와 **경쟁하지 않는다** |

그래서 **표본이 많은 부위와 적은 부위를 둘 다** 넣어야 방향이 갈린다.

| 부위 | n | 성격 |
|---|---|---|
| `IMI` 하벽 | 2,676 | 풍부 · 전두면 |
| `ASMI` 전중격 | 2,357 | 풍부 · 횡단면 |
| `ILMI` 하측벽 | 478 | 중간 |
| `LMI` 측벽 | 201 | **희소** (실험15에서 동작점이 못 쓸 수준이었다: 특이도 0.510) |

## 공정성 — **출력층 크기 말고는 전부 같게**

백본·에폭·겹·시드·전처리·유도 구성 전부 실험15와 **글자 그대로 동일**하고,
마지막 층만 `Dense(7, sigmoid)` → `Dense(1, sigmoid)` 로 바꾼다.
공유 쪽은 **재학습하지 않고 실험15의 arm 을 읽어 쓴다**(G0 관문이 정렬을 확인한다).

## 사전등록 (결과 보기 전에 고정)

| | 예측 | 성격 |
|---|---|---|
| **G0** | 실험15의 부위 목록·순서가 여기서 다시 계산한 것과 **일치** | 재사용 정렬 확인. 어긋나면 비교가 통째로 거짓 |
| **P-1 ★** | **희소 부위**(`LMI`·`ILMI`)에서 공유 ≥ 개별, CI가 0 제외 | 전이가 실재하는가 — 공유의 존재 이유 |
| **P-2** | **풍부 부위**(`IMI`·`ASMI`)에서 공유가 개별 대비 **−0.02 이내**(AUPRC) | 공유의 대가가 감당 가능한가 |
| **P-3** | 공유의 이득이 **희소 군에서 풍부 군보다 크다**(두 군 평균의 차, CI가 0 제외) | 기전 확인. **엄격한 단조는 요구하지 않는다** — AUPRC 는 유병률에 따라 척도가 달라 기전이 실재해도 4개 점이 순서대로 서지 않는다(픽스처로 확인) |

**판정은 전부 CI 기반 3분**(지지/기각/**미결**)이다. 점추정 2분 채점은 하지 않는다
— 실험13b·14에서 같은 실수를 두 번 했다.

### 결과가 무엇을 정하나

| P-1 | P-2 | 2단계 설계 |
|---|---|---|
| ✅ | ✅ | **헤드 하나에 소견을 모은다.** 전이는 실재하고 대가는 작다 |
| ✅ | ❌ | **혼합**: 흔한 소견은 전용 헤드, 희소 소견은 공유 헤드 |
| ❌ | — | **소견별로 나눈다.** 공유의 근거가 없다 |

## 비용

**다운로드 0 · 캐시 재사용.** 학습은 4부위 × 2구성 × 5겹 = **40회**.
실험15가 10회에 669초였으므로 **40분 안팎**으로 예상한다(첫 학습 후 노트북이 ETA를 찍는다).
겹별 체크포인트가 있어 중간에 끊겨도 이어서 간다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/ecg_preflight.py 인라인)
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 함수 적재: assert_label_vocab · decide · boot_indices")

In [ ]:
# CELL 1 — 설정 + 실험15 산출물 연결
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15와 **한 글자도 달라선 안 되는** 블록
CONFIGS = {"I+II": [0, 1], "12": list(range(12))}
K_FOLD, N_SEEDS, EPOCHS, SEED0, BOOT, NMIN = 5, 1, 20, 20260801, 2000, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
SITE_PLANE = {"IMI": "전두면", "ILMI": "혼합", "IPMI": "혼합", "IPLMI": "혼합",
              "ASMI": "횡단면", "AMI": "횡단면", "ALMI": "혼합",
              "LMI": "혼합", "PMI": "횡단면"}
# ★★ 여기까지

# 개별 학습할 대표 부위 — 풍부 2 + 중간 1 + 희소 1
SOLO = ["IMI", "ASMI", "ILMI", "LMI"]
RICH, SCARCE = ["IMI", "ASMI"], ["ILMI", "LMI"]
MARGIN = 0.02          # P-2: 풍부 부위에서 허용하는 공유의 대가

CONFIG = dict(exp="exp15c_shared_vs_percohort", quest="ailab-2026-0015",
              parent_exp="exp15_mi_loc_head",
              purpose="다중라벨 공유 헤드 vs 부위별 개별 이진 분류기 — 2단계 설계를 가른다",
              fairness="출력층 크기(Dense 7 → Dense 1) 말고는 백본·에폭·겹·시드 전부 동일",
              reuse="공유 arm 은 실험15에서 읽어 쓴다(재학습 없음)",
              solo_sites=SOLO, rich=RICH, scarce=SCARCE, margin=MARGIN,
              predictions={"G0": "실험15의 부위 목록·순서와 일치",
                           "P-1": "희소 부위에서 공유 >= 개별, CI가 0 제외 [주가설]",
                           "P-2": f"풍부 부위에서 공유가 개별 대비 -{MARGIN} 이내",
                           "P-3": "공유의 이득이 희소할수록 크다(n 역순 단조)"},
              scoring="전부 CI 기반 3분(지지/기각/미결). 점추정 2분 금지",
              k_fold=K_FOLD, n_seeds=N_SEEDS, epochs=EPOCHS, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp15c_shared_vs_solo", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
PREV15 = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp15_mi_loc_head" and os.path.isdir(r.get("dir", "")):
        PREV15 = r["dir"]
if PREV15 is None:
    raise RuntimeError("registry.jsonl에서 exp15_mi_loc_head를 못 찾았습니다 — 실험15를 먼저 돌리세요")
run.log(f"실험15 산출물: {PREV15}")
PREV_RESULT = json.load(open(os.path.join(PREV15, "result.json"), encoding="utf-8"))
SITES15 = PREV_RESULT["sites"]
run.log(f"  실험15 부위 {len(SITES15)}개: {SITES15}")

def prev_arm15(name):
    p = os.path.join(PREV15, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

missing = [f"{c}_f{k}" for c in CONFIGS for k in range(K_FOLD)
           if prev_arm15(f"{c}_f{k}") is None]
if missing:
    raise RuntimeError(f"실험15 arm 없음: {missing[:5]} …")
run.log(f"✅ 공유 arm {len(CONFIGS)*K_FOLD}개 확인 — 공유 쪽은 재학습하지 않습니다")

In [ ]:
# CELL 2 — 캐시 재사용 (다운로드 없음) + 라벨
import pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
for f in ("ptbxl_database.csv", "scp_statements.csv"):
    d = os.path.join(PTB, f)
    if not (os.path.exists(d) and os.path.getsize(d) > 0):
        subprocess.run(["wget", "-q", "-O", d, f"{BASE}/{f}"])
df = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")

CACHE = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"전량 캐시가 없습니다: {CACHE} — 실험15 CELL 2 를 먼저 돌리세요")
z = np.load(CACHE, allow_pickle=True)
X, FOLD10, EID = z["X"], z["fold"], z["eid"]
CV = (FOLD10 - 1) % K_FOLD
dfa = df.loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
run.log(f"캐시 재사용 X{X.shape} · 레코드 {len(EID):,} (다운로드 없음)")

# ── 라벨 어휘 검증 (실험13·13b·15 에서 세 번 틀렸던 자리)
vocab = {c for cs in dfa.codes for c in cs}
counts = {s: int(sum(s in cs for cs in dfa.codes)) for s in SITE_CANDIDATES}
assert_label_vocab(SITE_CANDIDATES, vocab, kind="scp 코드", counts=counts, min_count=NMIN)
run.log("✅ 라벨 어휘 검증 통과 (요청한 코드가 전부 데이터에 실재)")

SITES = [s for s in SITE_CANDIDATES if counts[s] >= NMIN]
Ymul = np.stack([[s in c for s in SITES] for c in dfa.codes]).astype("float32")

# ── 【G0】 실험15와 부위 목록·순서가 같은가 (다르면 arm 열 정렬이 깨진다)
run.log("\n【G0】 실험15 재사용 정렬")
run.log(f"  이번:   {SITES}")
run.log(f"  실험15: {SITES15}")
if SITES != SITES15:
    raise RuntimeError("부위 목록·순서가 실험15와 다릅니다 — arm 의 열 정렬이 깨져 비교가 무의미합니다")
run.log("  ✅ 일치 — 같은 열 위에서 비교합니다")

SHARED = {}
for c in CONFIGS:
    arr = np.zeros((len(EID), len(SITES)), "float32")
    for k in range(K_FOLD):
        arr[np.where(CV == k)[0]] = prev_arm15(f"{c}_f{k}")
    SHARED[c] = arr
assert_label_vocab(SOLO, set(SITES), kind="개별 학습 대상")
run.log(f"\n개별 학습할 부위 {SOLO} · "
        + " · ".join(f"{s} n={counts[s]:,}" for s in SOLO))

In [ ]:
# CELL 3 — 개별(이진) 학습: 4부위 × 2구성 × 5겹 = 40회
import tensorflow as tf
from tensorflow.keras import layers, models

def mask_of(cfg):
    m = np.zeros(12, "float32"); m[CONFIGS[cfg]] = 1.0
    return m

def build_head(seed, n_out):
    """★ 실험15의 build_head 와 **출력 차원 말고는 동일**. 시드 공식도 같다."""
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], 12))
    x = si
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si, layers.Dense(n_out, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")
    return m

SOLO_OOF = {f"{s}|{c}": np.zeros(len(EID), "float32") for s in SOLO for c in CONFIGS}
t0, done, total = time.time(), 0, len(SOLO) * len(CONFIGS) * K_FOLD * N_SEEDS
for s in SOLO:
    j = SITES.index(s)
    y1 = Ymul[:, j:j + 1]
    for c in CONFIGS:
        mk = mask_of(c)
        for k in range(K_FOLD):
            arm = f"solo_{s}_{c}_f{k}"
            cached = run.load_arm(arm)
            if cached is not None:
                SOLO_OOF[f"{s}|{c}"][np.where(CV == k)[0]] = cached.ravel()
                done += N_SEEDS; continue
            te = np.where(CV == k)[0]; rest = np.where(CV != k)[0]
            rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
            n_val = max(int(len(rest) * 0.12), 200)
            va, tr = rest[:n_val], rest[n_val:]
            acc = np.zeros((len(te), 1), "float32")
            for sd in range(N_SEEDS):
                # ★ 실험15와 **같은 시드 공식** — 초기화 차이를 비교에 섞지 않는다
                m = build_head(SEED0 + 100 * k + 15 + sd, 1)
                m.fit(X[tr] * mk, y1[tr], validation_data=(X[va] * mk, y1[va]),
                      epochs=EPOCHS, batch_size=128, verbose=0)
                acc += m.predict(X[te] * mk, batch_size=512, verbose=0)
                tf.keras.backend.clear_session(); done += 1
                if done == 1:
                    per = time.time() - t0
                    run.log(f"  ⏱ 첫 학습 {per:.0f}s → 전체 {total}회 예상 **{per*total/60:.0f}분**")
            run.save_arm(arm, acc / N_SEEDS)
            SOLO_OOF[f"{s}|{c}"][te] = (acc / N_SEEDS).ravel()
        run.log(f"  {s} · {c} 완료 ({done}/{total} · {time.time()-t0:.0f}s)")
run.log(f"\n총 {time.time()-t0:.0f}s")

In [ ]:
# CELL 4 — 평가 + 사전등록 채점
from sklearn.metrics import average_precision_score, roc_auc_score

def spec_at_sens(score, pos, target=0.90):
    p, n = score[pos], score[~pos]
    thr = float(np.quantile(p, 1.0 - target, method="lower"))
    return float((p >= thr).mean()), float((n < thr).mean()), float((score >= thr).mean())

q = lambda v, p: float(np.percentile(v, p))

def boot_ap_diff(a, b, y, seed=SEED0):
    """AUPRC(a) − AUPRC(b). 같은 재표본 축을 쓴다(짝지은 비교).

    ★ 유효하지 않은 반복은 **건너뛰지 않고 nan** 으로 남긴다. 길이를 고정해야
      나중에 여러 부위의 같은 반복끼리 묶어 군 평균의 CI를 낼 수 있다.
    """
    out = np.full(BOOT, np.nan)
    for t, i in enumerate(boot_indices(len(y), BOOT, seed)):
        if y[i].sum() < 5:
            continue
        out[t] = (average_precision_score(y[i], a[i])
                  - average_precision_score(y[i], b[i]))
    v = out[~np.isnan(out)]
    return float(v.mean()), q(v, 2.5), q(v, 97.5), out

R, BS = {}, {}
run.log("\n" + "=" * 116)
run.log("【공유 vs 개별】 부위별 AUPRC  (양수 Δ = 공유가 낫다)")
run.log("=" * 116)
run.log(f"  {'부위':<7}{'n':>6}{'구성':<7}{'공유':>9}{'개별':>9}{'Δ(공유−개별)':>14}{'95% CI':>22}"
        f"{'   공유 특이도':>12}{'개별 특이도':>12}")
for s in SOLO:
    j = SITES.index(s); y = Ymul[:, j].astype(bool)
    for c in CONFIGS:
        sh, so = SHARED[c][:, j], SOLO_OOF[f"{s}|{c}"]
        ap_sh = float(average_precision_score(y, sh))
        ap_so = float(average_precision_score(y, so))
        d, lo, hi, bs = boot_ap_diff(sh, so, y)
        BS[f"{s}|{c}"] = bs
        _, sp_sh, al_sh = spec_at_sens(sh, y)
        _, sp_so, al_so = spec_at_sens(so, y)
        R[f"{s}|{c}"] = {"site": s, "config": c, "n": int(y.sum()),
                         "auprc_shared": ap_sh, "auprc_solo": ap_so,
                         "delta": d, "ci": [lo, hi],
                         "auroc_shared": float(roc_auc_score(y, sh)),
                         "auroc_solo": float(roc_auc_score(y, so)),
                         "op_shared": {"spec": sp_sh, "alarm": al_sh},
                         "op_solo": {"spec": sp_so, "alarm": al_so}}
        star = "★" if lo > 0 else ("✗" if hi < 0 else " ")
        run.log(f"  {s:<7}{int(y.sum()):>6}{c:<7}{ap_sh:>9.3f}{ap_so:>9.3f}"
                f"{d:>+14.4f}   [{lo:+.4f}, {hi:+.4f}] {star}"
                f"{sp_sh:>12.3f}{sp_so:>12.3f}")
    run.log("  " + "-" * 112)

# ── 사전등록 채점 (전부 CI 3분)
run.log("\n" + "=" * 116)
run.log("【사전등록 채점】")
run.log("=" * 116)

def group_ci(sites, c):
    """군 평균 Δ 의 부트스트랩 CI — **같은 반복끼리 묶어** 평균을 낸다.
    (CI 상·하한을 평균 내는 건 CI가 아니다. 반복 축을 살려둔 이유가 이것이다.)"""
    M = np.stack([BS[f"{s}|{c}"] for s in sites])          # (부위, BOOT)
    keep = ~np.isnan(M).any(0)                              # 전 부위에서 유효한 반복만
    g = M[:, keep].mean(0)
    return float(g.mean()), q(g, 2.5), q(g, 97.5), int(keep.sum())

def group_decide(sites, thr, direction):
    """구성별로 판정하고, 두 구성이 엇갈리면 미결."""
    verdicts, stats = {}, {}
    for c in CONFIGS:
        m, lo, hi, nb = group_ci(sites, c)
        stats[c] = {"mean": m, "ci": [lo, hi], "n_boot": nb}
        verdicts[c] = decide(lo, hi, thr, direction)
    vs = set(verdicts.values())
    return (verdicts[list(CONFIGS)[0]] if len(vs) == 1 else None), verdicts, stats

P1, v1, st1 = group_decide(SCARCE, 0.0, ">")
run.log(f"  P-1 ★ 희소 부위{SCARCE} 공유 ≥ 개별 → {MARK[P1]}")
for c in CONFIGS:
    ds = [R[f"{s}|{c}"] for s in SCARCE]
    run.log(f"      {c:<7} " + " · ".join(
        f"{x['site']} Δ={x['delta']:+.4f} [{x['ci'][0]:+.4f},{x['ci'][1]:+.4f}]" for x in ds)
        + f"  ‖ 군평균 {st1[c]['mean']:+.4f} [{st1[c]['ci'][0]:+.4f},{st1[c]['ci'][1]:+.4f}]"
        + f" → {MARK[v1[c]]}")

P2, v2, st2 = group_decide(RICH, -MARGIN, ">")
run.log(f"  P-2  풍부 부위{RICH} 공유가 −{MARGIN} 이내 → {MARK[P2]}")
for c in CONFIGS:
    ds = [R[f"{s}|{c}"] for s in RICH]
    run.log(f"      {c:<7} " + " · ".join(
        f"{x['site']} Δ={x['delta']:+.4f} [{x['ci'][0]:+.4f},{x['ci'][1]:+.4f}]" for x in ds)
        + f"  ‖ 군평균 {st2[c]['mean']:+.4f} [{st2[c]['ci'][0]:+.4f},{st2[c]['ci'][1]:+.4f}]"
        + f" → {MARK[v2[c]]}")

# ── P-3: 공유의 이득이 **희소 쪽에서 더 큰가** (군 평균의 차, 짝지은 부트스트랩)
#    ★ 개별 부위 Δ 의 '엄격한 단조'를 요구하지 않는다. AUPRC 는 유병률에 따라 척도가
#      달라서, 기전이 실재해도 4개 점이 순서대로 서지 않는다(픽스처로 확인했다).
#      그래서 같은 방식으로 잰 **두 군의 평균 차**를 본다.
order = sorted(SOLO, key=lambda s: R[f"{s}|12"]["n"])
P3s, st3 = {}, {}
for c in CONFIGS:
    Ms = np.stack([BS[f"{s}|{c}"] for s in SCARCE])
    Mr = np.stack([BS[f"{s}|{c}"] for s in RICH])
    keep = ~np.isnan(np.vstack([Ms, Mr])).any(0)
    g = Ms[:, keep].mean(0) - Mr[:, keep].mean(0)
    lo, hi = q(g, 2.5), q(g, 97.5)
    st3[c] = {"mean": float(g.mean()), "ci": [lo, hi]}
    P3s[c] = decide(lo, hi, 0.0, ">")
    run.log(f"  P-3  {c:<7} 희소 − 풍부 = {g.mean():+.4f} [{lo:+.4f}, {hi:+.4f}] → {MARK[P3s[c]]}")
    run.log(f"        (참고) n 오름차순 {order} 의 Δ = "
            + " ".join(f"{R[f'{s}|{c}']['delta']:+.3f}" for s in order)
            + "  ※ AUPRC 척도가 유병률에 의존하므로 순서 자체는 판정에 쓰지 않는다")
vs3 = set(P3s.values())
P3 = P3s[list(CONFIGS)[0]] if len(vs3) == 1 else None
run.log(f"  P-3 → {MARK[P3]}")

if P1 is True and P2 is True:
    verdict = ("공유 채택 — 희소 부위에서 전이가 실재하고(P-1) 풍부 부위의 대가도 허용치 안(P-2). "
               "**2단계에서 소견을 헤드 하나에 모은다**")
elif P1 is True:
    verdict = ("혼합 설계 — 희소 부위는 공유가 낫지만 풍부 부위가 대가를 치른다. "
               "**흔한 소견은 전용 헤드, 희소 소견은 공유 헤드**로 나눈다")
elif P1 is False:
    verdict = ("개별 채택 — 희소 부위에서도 공유가 낫지 않다. 전이의 근거가 없으므로 "
               "**소견별로 나눈다**. 실험15의 다중라벨 구성은 편의였지 이득이 아니었다")
else:
    verdict = ("미결 — CI가 0을 걸친다. 공유·개별의 차이가 이 표본에서 검출되지 않으므로 "
               "**더 단순한 쪽(공유 하나)** 을 잠정 채택하고 소견 수를 늘려 재검정한다")
run.log(f"\n▶ {verdict}")
run.log("=" * 116)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(CONFIGS), figsize=(6.0 * len(CONFIGS), 3.8), squeeze=False)
for ax, c in zip(axes[0], CONFIGS):
    ds = [R[f"{s}|{c}"] for s in order]
    ys = np.arange(len(ds))
    ax.barh(ys, [x["delta"] for x in ds],
            xerr=[[x["delta"] - x["ci"][0] for x in ds], [x["ci"][1] - x["delta"] for x in ds]],
            capsize=4, color=["#36c" if x["site"] in SCARCE else "#888" for x in ds])
    ax.axvline(0, c="k", lw=.9); ax.axvline(-MARGIN, ls="--", c="r", lw=1)
    ax.set_yticks(ys); ax.set_yticklabels([f"{x['site']} (n={x['n']:,})" for x in ds], fontsize=8)
    ax.set_xlabel("Δ AUPRC (공유 − 개별) → 양수면 공유가 낫다")
    ax.set_title(f"{c} · 파랑=희소 부위", fontsize=10)
plt.tight_layout(); run.save_fig("shared_vs_solo", fig); plt.show()

run.save_json("evaluation", {"result": R, "P-1": P1, "P-2": P2, "P-3": P3,
                             "per_config": {"P-1": v1, "P-2": v2, "P-3": P3s},
                             "group_stats": {"scarce": st1, "rich": st2, "scarce_minus_rich": st3},
                             "verdict": verdict})

result = {"week": 2, "exp_id": "exp15c_shared_vs_solo", "quest": "ailab-2026-0015",
          "task": "다중라벨 공유 헤드 vs 부위별 개별 이진 분류기 — 2단계 라벨 트리 설계를 가른다",
          "split": "inter", "metric": "mean_auprc_shared_minus_solo_scarce_12lead",
          "value": round(float(np.mean([R[f"{s}|12"]["delta"] for s in SCARCE])), 4),
          "passed": bool(P1 is True and P2 is True),
          "date": time.strftime("%Y-%m-%d"), "k_fold": K_FOLD, "n_seeds": N_SEEDS,
          "solo_sites": SOLO, "rich": RICH, "scarce": SCARCE,
          "subgroups": R, "P-1": P1, "P-2": P2, "P-3": P3, "verdict": verdict,
          "summary": (f"희소{SCARCE} Δ(공유−개별) "
                      + " · ".join(f"{s} {R[f'{s}|12']['delta']:+.4f}" for s in SCARCE)
                      + f" · P-1 {MARK[P1]} P-2 {MARK[P2]} P-3 {MARK[P3]} · "
                      + verdict.split(' —')[0])}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp15c_shared_vs_percohort.ipynb \\
      --quest ailab-2026-0015 --step "exp15c-shared-vs-percohort" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 이 실험이 정하는 것

2단계(라벨 트리 전체)에서 소견이 수십 개로 늘어난다. 그때 **헤드 하나에 다 넣을지,
나눌지**를 여기서 정한다. 지금 정하지 않으면 나중에 수십 개를 다 학습해놓고 뒤집어야 한다.

## 결과 읽는 법

| P-1 | P-2 | 결론 |
|---|---|---|
| ✅ | ✅ | **공유 채택.** 소견을 헤드 하나에 모은다 |
| ✅ | ❌ | **혼합.** 흔한 소견은 전용 헤드, 희소 소견은 공유 |
| ❌ | — | **개별.** 실험15의 다중라벨은 편의였지 이득이 아니었다 |
| ⚠️ | — | 차이가 검출 안 됨 → **단순한 쪽(공유)** 잠정 채택, 소견 수 늘려 재검정 |

**P-1이 기각돼도 실험15의 결과는 유효하다.** 실험15는 "부위를 배우면 낫다"를 보였고
(그건 P-2·P-3로 확증됐다), 이 실험은 "어떻게 배울까"만 묻는다.

## 한계

- **대표 4부위만** 개별 학습한다(비용). 소견 수십 개로 갔을 때도 같은 방향인지는
  별도 확인이 필요하다 — 특히 **공유 헤드가 커질수록 경쟁이 심해진다.**
- `n_seeds: 1`. 공유·개별의 차가 작으면 재학습 잡음과 구분이 어렵다.
- 공유 쪽은 **7부위를 함께 배운 헤드**다. 개별과 비교할 때 "공유 대상 부위의 수"도
  같이 바뀌는 셈이라, 엄밀히는 **7-way 공유 vs 1-way** 의 비교다.
- **P-3은 두 군 평균의 차**로 본다. 개별 부위 Δ 의 순서는 참고용으로만 찍는다 —
  AUPRC 척도가 유병률에 의존해서 순서가 기전을 반영하지 않는다.
- **부위 간 Δ 의 직접 비교는 조심**해야 한다. 같은 Δ=0.05 도 유병률 0.11 인 부위와
  0.009 인 부위에서 뜻이 다르다. 그래서 판정은 **같은 군 안에서 평균**으로만 한다.
